### Re-ranking Hybrid Search Strategies
Re-ranking is a second state filtering process in retrieval systems, especially in RAG pipelines, where we:
1. First use a fast retriever (like BM25, FAISS, hybrid) to fetch top-k documents quickly.
2. Then use a more accurate but slower model (like a cross-encoder or LLM) to re-score and reorder those documents by relevance to the query.

-> It ensures that the most relevant documents appear at the top, improving the final answer from the LLM.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.schema.document import Document
from langchain_core.output_parsers import StrOutputParser

In [3]:
# Load the text file
loader = TextLoader("langchain_sample.txt", encoding="utf-8")
raw_docs = loader.load()


# Split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50, separators=["\n\n", ". ", "? ", "! "])  # split at paragraph or sentence endings
docs = text_splitter.split_documents(raw_docs)
for doc in docs:
    doc.page_content = doc.page_content.lstrip(". ").strip()

docs[:2]  # Display the first 2 chunks to verify

[Document(page_content='LangChain is an open-source framework designed to help developers build applications powered by large language models (LLMs) by providing structured, modular components that simplify tasks like prompt management, data retrieval, memory handling, and tool integration', metadata={'source': 'langchain_sample.txt'}),
 Document(page_content='At its core, LangChain acts as an orchestration layer that connects LLMs—such as those from OpenAI, Anthropic, or open-source providers—with external data sources, APIs, and workflows, enabling the creation of intelligent systems that go far beyond simple text generation', metadata={'source': 'langchain_sample.txt'})]

In [4]:
# User Query
query = "How can I use LangChain to build an application with memory and tools?"

In [5]:
# One way 
# FAISS and HuggingFace Embeddings for Dense Retrieval
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})
retriever

c:\RAG\env_langchain_lessthan1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\RAG\env_langchain_lessthan1\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C3868ABC10>, search_kwargs={'k': 8})

In [6]:
results = retriever.get_relevant_documents(query)
print("Dense Retriever Results: - HuggingFace Embeddings")
for idx, doc in enumerate(results):
    print(f"{idx + 1}. {doc.page_content} (Source: {doc.metadata['source']})")

Dense Retriever Results: - HuggingFace Embeddings
1. LangChain is an open-source framework designed to help developers build applications powered by large language models (LLMs) by providing structured, modular components that simplify tasks like prompt management, data retrieval, memory handling, and tool integration (Source: langchain_sample.txt)
2. In production scenarios, LangChain can be combined with orchestration tools, caching layers, and observability platforms to build reliable, high-performance systems, and it often works alongside frameworks like FastAPI or Flask to serve applications as APIs. Over time, LangChain has evolved into a broader ecosystem that includes tools for evaluation, debugging, and deployment, helping developers monitor how their LLM applications behave and improve them iteratively (Source: langchain_sample.txt)
3. Tools in LangChain are simply functions or APIs that the agent can invoke, and they can range from simple utilities to complex services like c

c:\RAG\env_langchain_lessthan1\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


In [7]:
# Another way
# FAISS and OpenAI Embeddings for Dense Retrieval
from langchain.embeddings import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1536)
vectorstore_openai = FAISS.from_documents(docs, embedding_model)
retriever_openai = vectorstore_openai.as_retriever(search_kwargs={"k": 8})
retriever_openai

c:\RAG\env_langchain_lessthan1\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(
c:\RAG\env_langchain_lessthan1\Lib\site-packages\langchain_community\embeddings\openai.py:268: UserWarning: WARNING! dimensions is not default parameter.
                    dimensions was transferred to model_kwargs.
                    Please confirm that dimensions is what you intended.
  warnings.warn(


VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C488715190>, search_kwargs={'k': 8})

In [8]:
results_openai = retriever_openai.get_relevant_documents(query)
print("Dense Retriever Results: - OpenAI Embeddings")
for idx, doc in enumerate(results_openai):
    print(f"{idx + 1}. {doc.page_content} (Source: {doc.metadata['source']})")

Dense Retriever Results: - OpenAI Embeddings
1. In production scenarios, LangChain can be combined with orchestration tools, caching layers, and observability platforms to build reliable, high-performance systems, and it often works alongside frameworks like FastAPI or Flask to serve applications as APIs. Over time, LangChain has evolved into a broader ecosystem that includes tools for evaluation, debugging, and deployment, helping developers monitor how their LLM applications behave and improve them iteratively (Source: langchain_sample.txt)
2. One of its foundational ideas is that LLMs alone are not sufficient for building robust applications because they lack real-time knowledge, long-term memory, and the ability to reliably perform multi-step reasoning or interact with external systems; LangChain addresses these limitations by introducing abstractions like chains, agents, retrievers, and memory modules (Source: langchain_sample.txt)
3. In essence, LangChain transforms the raw capab

In [9]:
# LLM
llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",  # correct argument in <1.0
    temperature=0.2
)



c:\RAG\env_langchain_lessthan1\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import ChatOpenAI`.
  warn_deprecated(


In [10]:
# Prompt and Use the LLM to Re-rank the retrieved documents
prompt = PromptTemplate.from_template("""
You are a helpful assitant. Your task is to rank the following documents from most to least relevant to the user's question.
User Question: "{question}"

Documents:
{documents}

Instructions:
- Think about the relevance of each document to the user's question.
- Return a list of document indices in ranked order, starting from the most relevant to the least relevant.

Output format: comma-separated list of document indices (e.g., "0, 2, 1" if document 0 is most relevant, followed by document 2, and then document 1).
""")


In [11]:
retriever_docs = retriever.invoke(query)
retriever_docs

[Document(page_content='LangChain is an open-source framework designed to help developers build applications powered by large language models (LLMs) by providing structured, modular components that simplify tasks like prompt management, data retrieval, memory handling, and tool integration', metadata={'source': 'langchain_sample.txt'}),
 Document(page_content='In production scenarios, LangChain can be combined with orchestration tools, caching layers, and observability platforms to build reliable, high-performance systems, and it often works alongside frameworks like FastAPI or Flask to serve applications as APIs. Over time, LangChain has evolved into a broader ecosystem that includes tools for evaluation, debugging, and deployment, helping developers monitor how their LLM applications behave and improve them iteratively', metadata={'source': 'langchain_sample.txt'}),
 Document(page_content='Tools in LangChain are simply functions or APIs that the agent can invoke, and they can range f

In [12]:
chain=prompt | llm | StrOutputParser()
chain

PromptTemplate(input_variables=['documents', 'question'], template='\nYou are a helpful assitant. Your task is to rank the following documents from most to least relevant to the user\'s question.\nUser Question: "{question}"\n\nDocuments:\n{documents}\n\nInstructions:\n- Think about the relevance of each document to the user\'s question.\n- Return a list of document indices in ranked order, starting from the most relevant to the least relevant.\n\nOutput format: comma-separated list of document indices (e.g., "0, 2, 1" if document 0 is most relevant, followed by document 2, and then document 1).\n')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001C48A96AFD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001C48A969510>, temperature=0.2, openai_api_key='sk-proj-d1cRBA3WP31JmoVaM69XmU3GJWHI1sg1ZXs0wrAwEwjTpbiZvjgeAevF-c5kSFwG1pC6OaKwaGT3BlbkFJnT7br8-h456WBdNrP5LQuJJaV6bwmVgTNDWbSxTqLvCJqV8lSz9e

In [13]:
doc_lines = [f"{i+1}. {doc.page_content}" for i, doc in enumerate(retriever_docs)]
formatted_docs = "\n".join(doc_lines)

In [14]:
doc_lines

['1. LangChain is an open-source framework designed to help developers build applications powered by large language models (LLMs) by providing structured, modular components that simplify tasks like prompt management, data retrieval, memory handling, and tool integration',
 '2. In production scenarios, LangChain can be combined with orchestration tools, caching layers, and observability platforms to build reliable, high-performance systems, and it often works alongside frameworks like FastAPI or Flask to serve applications as APIs. Over time, LangChain has evolved into a broader ecosystem that includes tools for evaluation, debugging, and deployment, helping developers monitor how their LLM applications behave and improve them iteratively',
 '3. Tools in LangChain are simply functions or APIs that the agent can invoke, and they can range from simple utilities to complex services like code execution environments or search engines. Another important aspect is output parsing, which ensure

In [15]:
response=chain.invoke({
    "question": query,
    "documents": formatted_docs
})
response

'1, 5, 8, 3, 4, 2, 6, 7'

In [16]:
# Parse and re-rank
ranked_indices = [int(idx.strip()) - 1 for idx in response.split(",") if idx.strip().isdigit()]
ranked_indices

[0, 4, 7, 2, 3, 1, 5, 6]

In [17]:
reranked_docs = [retriever_docs[idx] for idx in ranked_indices if 0 <= idx < len(retriever_docs)] # Reorder the original documents based on the ranked indices
print("\n Final Re-ranked Documents:")
for rank, idx in enumerate(ranked_indices):
    doc = retriever_docs[idx]
    print(f"{rank + 1}. {doc.page_content} (Source: {doc.metadata['source']})")


 Final Re-ranked Documents:
1. LangChain is an open-source framework designed to help developers build applications powered by large language models (LLMs) by providing structured, modular components that simplify tasks like prompt management, data retrieval, memory handling, and tool integration (Source: langchain_sample.txt)
2. One of its foundational ideas is that LLMs alone are not sufficient for building robust applications because they lack real-time knowledge, long-term memory, and the ability to reliably perform multi-step reasoning or interact with external systems; LangChain addresses these limitations by introducing abstractions like chains, agents, retrievers, and memory modules (Source: langchain_sample.txt)
3. Speaking of conversations, LangChain includes memory modules that store and manage past interactions, enabling chatbots to maintain context across multiple turns instead of treating each query independently; memory can be as simple as a list of previous messages or